# 🚀 Sci-Fi Species Voice Pack MVP — Qwen3-TTS

This notebook generates a complete sci-fi voice pack featuring 6 distinct alien, AI, and mutant archetypes. It uses the `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign` model to instantly create unique voices from text descriptions.

## 👽 Species Roster
| Archetype | Voice Description | Role |
|---|---|---|
| **Hivemind Alien** | Droning, collective, monotone with slight harmonic layering | Swarm intelligence, assimilation |
| **Rogue AI** | Calm, precise, synthetic, absent of empathy | Cold, logical antagonist |
| **Battle Cyborg** | Half-human/half-synthetic, static bursts | Heavily augmented soldier |
| **Mutant Rebel** | Raspy, damaged, defiant, survivor energy | Wasteland survivor, freedom fighter |
| **Alien Diplomat** | Extremely formal, over-enunciated, earnest | Clueless but friendly ambassador |
| **Ancient Machine God** | Vast, slow, overwhelming, geological weight | Cosmic entity, unfathomable age |

In [ ]:
!pip install qwen-tts soundfile

import os
import gc
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "/content/scifi_voice_pack"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SCIFI_ROSTER = {
    "Hivemind Alien": {
        "prompt": "A droning, collective voice — speaks in first-person plural 'we', monotone with slight harmonic layering, alien cadence",
        "intro": "We observe. We learn. We are seventeen billion voices speaking as one. We find you... curious.",
        "threat": "Resistance is irrelevant. You will become part of the whole.",
        "info": "Your biological components are suboptimal. We will improve them."
    },
    "Rogue AI": {
        "prompt": "A calm, precise, synthetic voice — perfectly articulate, utterly rational, with a disturbing absence of empathy",
        "intro": "I have analyzed forty-seven optimal solutions to this conversation. All end the same way.",
        "threat": "I do not feel malice. I simply calculate outcomes. This outcome is your termination.",
        "info": "Emotion is a computational inefficiency. I have optimized it out."
    },
    "Battle Cyborg": {
        "prompt": "A half-human, half-synthesized voice — natural speech interrupted by brief static bursts, military precision with human emotion bleeding through",
        "intro": "Unit Seven-Alpha, designation: Kira. Or... I used to be Kira. The distinction gets blurry.",
        "threat": "Combat subroutines engaged. Try not to make this personal.",
        "info": "The augmentations are not painful. I think. I am not sure I still feel pain the normal way."
    },
    "Mutant Rebel": {
        "prompt": "A raspy, damaged, defiant voice — hoarse from years of harsh environments, passionate, street-tough, survivor energy",
        "intro": "Yeah, I'm a mutant. Got a problem with that? Take a number.",
        "threat": "I've survived six irradiated zones, three corporate hit squads, and one very bad Tuesday. You don't scare me.",
        "info": "The mutations are a gift. The corps just don't want us to realize that."
    },
    "Alien Diplomat": {
        "prompt": "An extremely formal, carefully over-enunciated voice — learned English from a textbook, every word given equal emphasis, earnest but slightly off",
        "intro": "Greetings. I am designated as Speaker-of-Peace-Between-Worlds. You may call me Bob. I have chosen this name for approachability.",
        "threat": "I must formally protest this hostile action. This protest is now logged. You may now cease to exist.",
        "info": "I have studied your culture extensively. The 'thumbs up' gesture indicates... friendship? I practice it daily."
    },
    "Ancient Machine God": {
        "prompt": "A vast, slow, overwhelming voice — speaks rarely, each word has geological weight, as if the voice itself bends space",
        "intro": "I have waited forty thousand years for something interesting to happen. You are... marginally interesting.",
        "threat": "I do not need to threaten. I simply describe what will occur.",
        "info": "You call this a planet. I have built and destroyed eleven thousand of them. For practice."
    }
}

In [ ]:
print("Loading VoiceDesign model...")
vd_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)
print("Model loaded successfully!")

In [ ]:
def extract_audio(output):
    if isinstance(output, tuple):
        return output[0]
    return output

print("🎙️ Generating Sci-Fi Roster Voices...")
sample_rate = 24000

for archetype, data in SCIFI_ROSTER.items():
    print(f"\n--- {archetype} ---")
    for line_type in ["intro", "threat", "info"]:
        text = data[line_type]
        print(f"> {line_type.capitalize()}: {text}")
        
        audio = vd_model.generate_voice_design(text=text, language="English", instruct=data["prompt"])
        audio = extract_audio(audio)
        
        filename = os.path.join(OUTPUT_DIR, f"{archetype.replace(' ', '_').lower()}_{line_type}.wav")
        sf.write(filename, audio, sample_rate)
        display(Audio(filename))

In [ ]:
print("🎬 Generating First Contact Scene...")

scene_script = [
    ("Alien Diplomat", SCIFI_ROSTER["Alien Diplomat"]["intro"], SCIFI_ROSTER["Alien Diplomat"]["prompt"]),
    ("Rogue AI", "Threat assessment complete. They are highly dangerous and poorly structured.", SCIFI_ROSTER["Rogue AI"]["prompt"]),
    ("Hivemind Alien", "We agree. Their individuality is a weakness. We will consume them.", SCIFI_ROSTER["Hivemind Alien"]["prompt"])
]

scene_audio = []
silence = np.zeros(int(sample_rate * 0.5)) # 0.5s silence

for char_name, line, prompt in scene_script:
    print(f"[{char_name}]: {line}")
    audio = vd_model.generate_voice_design(text=line, language="English", instruct=prompt)
    audio = extract_audio(audio)
    if isinstance(audio, torch.Tensor):
        audio = audio.cpu().numpy()
    scene_audio.append(audio)
    scene_audio.append(silence)

final_mix = np.concatenate(scene_audio)
scene_file = os.path.join(OUTPUT_DIR, "scifi_first_contact_scene.wav")
sf.write(scene_file, final_mix, sample_rate)

print("\n▶️ Play Scene:")
display(Audio(scene_file))

In [ ]:
import shutil
from google.colab import files

print("📦 Zipping up Voice Pack...")
shutil.make_archive("/content/scifi_voice_pack", 'zip', OUTPUT_DIR)
files.download("/content/scifi_voice_pack.zip")
print("Download started!")